# Lecture 5 - Notebook 2: Feature Engineering und Model-ready Data

**Kurs:** Data Science - Vorlesung 5  
**Datei:** `lecture_05_notebook_02_feature_engineering_model_ready.ipynb`

In diesem Notebook entsteht aus dem bereinigten Datensatz ein **model-ready dataset**.

Dafür brauchen wir:

- eine Zielvariable `Y`
- mehrere Prädiktoren `X`
- numerische Features
- keine IDs als Prädiktoren
- nachvollziehbare neue Variablen

## Für Studierende, die nicht anwesend waren

Dieses Notebook setzt auf dem bereinigten Datensatz auf.

**Wichtig:** Der Schwerpunkt liegt nicht auf Python-Syntax, sondern auf der Frage: Wie wird aus einem bereinigten Datensatz ein Datensatz, den wir später für Analysen und Modelle verwenden können?

Die Arbeitslogik ist:

1. bereinigte Daten laden
2. Skalen und neue Features bilden
3. Summary Statistics prüfen
4. Zielvariable `Y` festlegen
5. Prädiktoren `X` vorbereiten
6. kategoriale Variablen für Modelle codieren
7. model-ready Datei speichern

## Dateien für dieses Notebook

Dieses Notebook verwendet normalerweise:

- `lecture_05_employee_survey_cleaned.csv` - Ergebnis aus Notebook 1

Falls diese Datei noch nicht existiert, verwendet das Notebook automatisch:

- `lecture_05_employee_survey_cleaned_reference.csv`

Am Ende erzeugt es:

- `lecture_05_employee_survey_model_ready.csv`

So können Sie Notebook 2 auch bearbeiten, wenn Sie Notebook 1 noch nicht vollständig abgeschlossen haben.

## Vor dem Start

Dieses Notebook erwartet normalerweise die Datei:

`lecture_05_employee_survey_cleaned.csv`

Diese Datei entsteht in Notebook 1.

Falls Sie Notebook 1 nicht ausgeführt haben, lädt dieses Notebook automatisch die Referenzdatei:

`lecture_05_employee_survey_cleaned_reference.csv`

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)

## 1. Bereinigte Daten laden

Wir arbeiten jetzt nicht mehr mit den Rohdaten, sondern mit dem bereinigten Datensatz.

**Wichtig:** Das ist der Übergang von Rohdaten zu analysis-ready data.

In [2]:
# Standardfall:
# Wir laden die Datei, die in Notebook 1 erzeugt wurde.
#
# Fallback:
# Falls diese Datei noch nicht existiert, verwenden wir die bereitgestellte Referenzdatei.

try:
    df = pd.read_csv("lecture_05_employee_survey_cleaned.csv")
    print("Geladen: lecture_05_employee_survey_cleaned.csv")
except FileNotFoundError:
    df = pd.read_csv("lecture_05_employee_survey_cleaned_reference.csv")
    print("Geladen: lecture_05_employee_survey_cleaned_reference.csv")

df.head()

Geladen: lecture_05_employee_survey_cleaned.csv


,participant_id,department,gender,age,weekly_hours,sleep_hours,commute_minutes,workload_1,workload_2,autonomy_1,autonomy_2,support_1,support_2,stress_score,motivation,satisfaction,performance_rating,absent_days
0,P001,HR,Male,42.0,45.4,8.6,21.0,4.0,4.0,3.0,4.0,3.0,2.0,54.0,3.3,3.2,3.1,4.0
1,P002,Operations,Male,22.0,45.3,5.8,30.0,4.0,4.0,4.0,3.0,5.0,5.0,33.0,4.0,4.1,4.5,3.0
2,P003,Marketing,Female,36.0,33.1,9.1,26.0,3.0,3.0,4.0,4.0,2.0,2.0,37.0,3.9,3.2,3.5,0.0
3,P004,Sales,Female,47.0,35.3,5.0,23.0,3.0,3.0,4.0,4.0,2.0,1.0,54.0,2.8,3.2,3.4,2.0
4,P005,Sales,Female,25.0,42.6,6.8,23.0,3.0,3.0,3.0,3.0,2.0,2.0,49.0,3.8,2.9,3.3,2.0


## 2. Kleine Skalen bilden

Manche Konstrukte werden mit mehreren Items gemessen.

Beispiel:

- `workload_1` und `workload_2` messen beide Arbeitsbelastung
- `autonomy_1` und `autonomy_2` messen beide Autonomie
- `support_1` und `support_2` messen beide Unterstützung

Für die Übung bilden wir einfache Mittelwerte.

In [3]:
# axis=1 bedeutet: Mittelwert zeilenweise berechnen.
# Pro Person entsteht also ein neuer Skalenwert.

df["workload_mean"] = df[["workload_1", "workload_2"]].mean(axis=1)
df["autonomy_mean"] = df[["autonomy_1", "autonomy_2"]].mean(axis=1)
df["support_mean"] = df[["support_1", "support_2"]].mean(axis=1)

df[["workload_mean", "autonomy_mean", "support_mean"]].head()

,workload_mean,autonomy_mean,support_mean
0,4.0,3.5,2.5
1,4.0,3.5,5.0
2,3.0,4.0,2.0
3,3.0,4.0,1.5
4,3.0,3.0,2.0


## 3. Eine eigene Feature-Idee

Ein Feature ist eine Variable, die wir bewusst für Analyse oder Modellierung erzeugen.

Hier erzeugen wir einen einfachen `overload_index`.

Die Idee:

- hohe Arbeitsbelastung erhöht den Index
- hoher Stress erhöht den Index
- mehr Schlaf senkt den Index

Das ist keine perfekte psychologische Skala. Es ist eine transparente Feature-Idee für die Übung.

In [4]:
# stress_score wird durch 25 geteilt, damit der Wert ungefähr zur 1-bis-5-Skala passt.
# sleep_hours wird durch 4 geteilt und abgezogen, weil mehr Schlaf den Belastungsindex senken soll.

df["overload_index"] = (
    df["workload_mean"]
    + df["stress_score"] / 25
    - df["sleep_hours"] / 4
)

df[["workload_mean", "stress_score", "sleep_hours", "overload_index"]].head()

,workload_mean,stress_score,sleep_hours,overload_index
0,4.0,54.0,8.6,4.010
1,4.0,33.0,5.8,3.870
2,3.0,37.0,9.1,2.205
3,3.0,54.0,5.0,3.910
4,3.0,49.0,6.8,3.260


## 4. Summary Statistics nach der Bereinigung

Jetzt sind beschreibende Statistiken sinnvoller, weil wir:

- Kategorien bereinigt haben
- ungültige Werte markiert haben
- fehlende Werte behandelt haben
- neue Variablen erzeugt haben

In [5]:
summary_cols = [
    "satisfaction", "stress_score", "motivation",
    "workload_mean", "autonomy_mean", "support_mean",
    "sleep_hours", "overload_index",
]

df[summary_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
satisfaction,80.0,3.54750,0.734153,1.80,3.07500,3.6000,4.1250,5.000
stress_score,80.0,42.67500,12.231703,14.00,35.00000,42.0000,49.0000,99.000
motivation,80.0,3.80375,0.492846,2.70,3.50000,3.8000,4.1000,4.900
workload_mean,80.0,2.99375,0.905307,1.00,2.50000,3.0000,3.5000,5.000
autonomy_mean,80.0,3.45000,0.731740,1.50,3.00000,3.5000,4.0000,5.000
support_mean,80.0,3.49375,0.905307,1.50,3.00000,3.5000,4.0000,5.000
sleep_hours,80.0,7.07500,0.815708,5.00,6.40000,7.0500,7.6000,9.100
overload_index,80.0,2.93200,1.278105,-0.39,2.20375,2.9675,3.9125,5.745


## 5. Gruppenvergleiche mit `groupby`

`groupby()` ist nützlich, wenn wir Kennzahlen nach Gruppen berechnen möchten.

Hier fragen wir:

> Wie unterscheidet sich die durchschnittliche Zufriedenheit nach Abteilung?

In [6]:
# agg() berechnet mehrere Kennzahlen gleichzeitig.
# round(2) rundet die Ausgabe auf zwei Nachkommastellen.

department_summary = (
    df.groupby("department")["satisfaction"]
    .agg(["count", "mean", "std"])
    .round(2)
)

department_summary

,count,mean,std
department,,,
HR,13,3.43,0.77
IT,14,3.81,0.54
Marketing,14,3.29,0.89
Operations,13,3.49,0.61
Sales,25,3.62,0.78
Unknown,1,4.00,NaN


## 6. Zielvariable Y und Prädiktoren X

Für die spätere Modellierung brauchen wir zwei Objekte:

- `y`: die abhängige Variable / Zielvariable
- `X`: die unabhängigen Variablen / Prädiktoren

In diesem Beispiel wählen wir:

- `satisfaction` als `Y`
- mehrere Arbeits- und Kontextvariablen als `X`

In [7]:
# Zielvariable
target = "satisfaction"

# Numerische Prädiktoren:
# Diese Variablen können direkt als Zahlen in X aufgenommen werden.
num_predictors = [
    "age", "weekly_hours", "sleep_hours", "commute_minutes",
    "workload_mean", "autonomy_mean", "support_mean",
    "stress_score", "motivation", "overload_index",
]

# Kategoriale Prädiktoren:
# Diese müssen gleich noch in numerische Dummy-Variablen übersetzt werden.
cat_predictors = ["department", "gender"]

y = df[target]
X_num = df[num_predictors]

print("Y:", y.shape)
print("X_num:", X_num.shape)

Y: (80,)
X_num: (80, 10)


## 7. Kategoriale Variablen dummy-codieren

Viele Modelle können nicht direkt mit Text wie `HR`, `Sales` oder `Female` rechnen.

`pd.get_dummies()` übersetzt Kategorien in 0/1-Spalten.

Beispiel:

- `department_Sales = 1` bedeutet: Person gehört zu Sales
- `department_Sales = 0` bedeutet: Person gehört nicht zu Sales

In [8]:
# drop_first=True entfernt eine Referenzkategorie.
# Das ist bei linearen Modellen oft sinnvoll, weil sonst eine redundante Spalte entsteht.
# dtype=int sorgt dafür, dass die Dummy-Spalten als 0/1-Zahlen gespeichert werden.

X_cat = pd.get_dummies(
    df[cat_predictors],
    drop_first=True,
    dtype=int,
)

X_cat.head()

,department_IT,department_Marketing,department_Operations,department_Sales,department_Unknown,gender_Male,gender_Nonbinary,gender_Prefer not to say,gender_Unknown
0,0,0,0,0,0,1,0,0,0
1,0,0,1,0,0,1,0,0,0
2,0,1,0,0,0,0,0,0,0
3,0,0,0,1,0,0,0,0,0
4,0,0,0,1,0,0,0,0,0


## 8. Feature-Matrix X zusammensetzen

Jetzt kombinieren wir:

- numerische Prädiktoren
- dummy-codierte kategoriale Prädiktoren

Das Ergebnis ist unsere Feature-Matrix `X`.

In [9]:
# concat(..., axis=1) fügt Spalten nebeneinander zusammen.
# Jede Zeile in X muss zur gleichen Person gehören wie die entsprechende Zeile in y.

X = pd.concat([X_num, X_cat], axis=1)

print("Y:", y.shape)
print("X:", X.shape)

X.head()

Y: (80,)
X: (80, 19)


,age,weekly_hours,sleep_hours,commute_minutes,workload_mean,autonomy_mean,support_mean,stress_score,motivation,overload_index,department_IT,department_Marketing,department_Operations,department_Sales,department_Unknown,gender_Male,gender_Nonbinary,gender_Prefer not to say,gender_Unknown
0,42.0,45.4,8.6,21.0,4.0,3.5,2.5,54.0,3.3,4.010,0,0,0,0,0,1,0,0,0
1,22.0,45.3,5.8,30.0,4.0,3.5,5.0,33.0,4.0,3.870,0,0,1,0,0,1,0,0,0
2,36.0,33.1,9.1,26.0,3.0,4.0,2.0,37.0,3.9,2.205,0,1,0,0,0,0,0,0,0
3,47.0,35.3,5.0,23.0,3.0,4.0,1.5,54.0,2.8,3.910,0,0,0,1,0,0,0,0,0
4,25.0,42.6,6.8,23.0,3.0,3.0,2.0,49.0,3.8,3.260,0,0,0,1,0,0,0,0,0


## 9. Was gehört nicht in X?

`participant_id` ist eine ID.  
Sie hilft beim Verwalten von Daten, aber sie ist normalerweise kein sinnvoller Prädiktor.

Auch die Zielvariable selbst darf nicht versehentlich in `X` landen.

In [10]:
# Diese beiden Checks sollten False sein.
# Wenn hier True steht, ist etwas in der Feature-Matrix falsch.

print("participant_id in X:", "participant_id" in X.columns)
print("satisfaction in X:", "satisfaction" in X.columns)

participant_id in X: False
satisfaction in X: False


## 10. Optional: sehr kleiner Modelltest

Dieser Schritt ist nicht der Schwerpunkt der Vorlesung.

Er zeigt nur:

> Sobald `Y` und `X` sauber vorbereitet sind, ist Modellierung technisch der nächste Schritt.

Die Interpretation bleibt hier bewusst kurz.

In [11]:
# Für eine lineare Regression brauchen wir eine Intercept-Spalte.
# np.ones(len(X)) erzeugt eine Spalte aus Einsen.
# Diese Spalte übernimmt die Rolle der Konstanten in der Regressionsgleichung.

X_design = np.column_stack([
    np.ones(len(X)),
    X.astype(float).to_numpy(),
])

# y wird in ein NumPy-Array umgewandelt.
y_array = y.to_numpy()

# np.linalg.lstsq berechnet die Koeffizienten nach der Methode der kleinsten Quadrate.
# Das ist hier nur ein kurzer technischer Test.
coef, residuals, rank, s = np.linalg.lstsq(X_design, y_array, rcond=None)

# Vorhersagewerte berechnen
y_hat = X_design @ coef

# RMSE als einfache Fehlerkennzahl
rmse = np.sqrt(np.mean((y_array - y_hat) ** 2))

print("Anzahl Koeffizienten:", len(coef))
print("RMSE:", round(rmse, 3))

Anzahl Koeffizienten: 20
RMSE: 0.379


## 11. Model-ready Datensatz speichern

Zum Schluss speichern wir einen Datensatz, der direkt für spätere Modellierungsübungen genutzt werden kann.

Er enthält:

- `participant_id` zur Orientierung
- `satisfaction` als Zielvariable
- alle vorbereiteten Prädiktoren

In [12]:
model_ready = pd.concat([df[["participant_id", target]], X], axis=1)

model_ready.to_csv("lecture_05_employee_survey_model_ready.csv", index=False)

model_ready.head()

,participant_id,satisfaction,age,weekly_hours,sleep_hours,commute_minutes,workload_mean,autonomy_mean,support_mean,stress_score,motivation,overload_index,department_IT,department_Marketing,department_Operations,department_Sales,department_Unknown,gender_Male,gender_Nonbinary,gender_Prefer not to say,gender_Unknown
0,P001,3.2,42.0,45.4,8.6,21.0,4.0,3.5,2.5,54.0,3.3,4.010,0,0,0,0,0,1,0,0,0
1,P002,4.1,22.0,45.3,5.8,30.0,4.0,3.5,5.0,33.0,4.0,3.870,0,0,1,0,0,1,0,0,0
2,P003,3.2,36.0,33.1,9.1,26.0,3.0,4.0,2.0,37.0,3.9,2.205,0,1,0,0,0,0,0,0,0
3,P004,3.2,47.0,35.3,5.0,23.0,3.0,4.0,1.5,54.0,2.8,3.910,0,0,0,1,0,0,0,0,0
4,P005,2.9,25.0,42.6,6.8,23.0,3.0,3.0,2.0,49.0,3.8,3.260,0,0,0,1,0,0,0,0,0


## Abschluss-Check

Nach diesem Notebook sollten Sie erklären können:

- warum aus Rohdaten erst analysis-ready data werden
- was Feature Engineering bedeutet
- wie aus einer Kategorie Dummy-Variablen entstehen
- warum IDs nicht in `X` gehören
- wie `Y` und `X` in pandas entstehen